# Season split and baselines

| Split | Seasons | Games |
|---|---|---|
| Train | 2014-15 to 2019-20 | 7,209 |
| Validation | 2020-21, 2021-22 | 2,310 |
| Test (scored once, on day 10) | 2022-23 to 2024-25 | 3,690 |

The three baselines below are scored on validation. Every model on days 7 to 9 has to beat Elo on log loss, with a paired bootstrap interval, before it counts as better.

In [1]:
import json
import os
from pathlib import Path

os.environ.setdefault("NBA_DATA_DIR", "../data")

import numpy as np
import pandas as pd

from nba_predictor.evaluate import log_loss, split

feats = pd.read_parquet(Path(os.environ["NBA_DATA_DIR"]) / "processed" / "features.parquet")
train, validation, test = split(feats)
print(len(train), len(validation), len(test))
report = json.loads(Path("../reports/baselines.json").read_text())
cols = [
    "log_loss",
    "log_loss_95",
    "brier",
    "accuracy",
    "accuracy_95",
    "auc",
    "mean_predicted",
    "log_loss_minus_elo_95",
]
pd.DataFrame(report["baselines"]).T[cols]

7209 2310 3690


,log_loss,log_loss_95,brier,accuracy,accuracy_95,auc,mean_predicted,log_loss_minus_elo_95
always home,0.6919,"[0.6852, 0.6983]",0.2493,0.5437,"[0.5234, 0.5645]",0.5,0.579,"[0.0372, 0.0641]"
better record (log5),0.6525,"[0.6402, 0.6648]",0.2301,0.6108,"[0.5905, 0.6303]",0.6645,0.4988,"[0.0021, 0.0205]"
Elo,0.6411,"[0.6268, 0.6561]",0.2248,0.6359,"[0.6156, 0.6545]",0.6841,0.5775,"[0.0, 0.0]"


- **Elo is the baseline to beat:** log loss 0.641 (0.627 to 0.656), 63.6% right.
- **Better record (log5) is close behind** but measurably worse: 0.002 to 0.021 more log loss on the same games. It ignores home advantage and knows nothing on opening night.
- **Always home barely beats a coin flip** (0.692 against 0.693). It predicts 57.9%, the training seasons' rate, but home teams won 54.4% in these two seasons.

## Elo on validation: the home edge it learned is too big

In [2]:
pd.DataFrame(
    {
        "home won": validation.groupby("season")["home_win"].mean(),
        "Elo predicted": validation.groupby("season")["elo_prob"].mean(),
    }
).round(4)

,home won,Elo predicted
season,,
2020-21,0.5435,0.5765
2021-22,0.5439,0.5783


In [3]:
# Diagnostic only, fitted on validation itself: the best possible constant shift of
# Elo's log-odds. It shows how much of Elo's loss is the home-edge mismatch, not a model.
z = np.log(validation["elo_prob"] / (1 - validation["elo_prob"]))
shifts = np.linspace(-0.4, 0.2, 61)
losses = [log_loss(validation["home_win"], 1 / (1 + np.exp(-(z + s)))) for s in shifts]
best = shifts[int(np.argmin(losses))]
print(
    f"best shift {best:+.2f} log-odds: log loss {min(losses):.4f} against "
    f"{log_loss(validation['home_win'], validation['elo_prob']):.4f} unshifted"
)

best shift -0.15 log-odds: log loss 0.6385 against 0.6411 unshifted


## Findings

1. **Elo overrates home teams by about 3.4 points on validation** (57.8% predicted, 54.4% won, the same in both seasons). This is the day 3 warning: its 60-point home bonus was tuned when home teams won 58%.
2. **But it costs little.** Even the best possible shift, fitted on validation itself, only improves log loss by about 0.003. Most of Elo's error is in which team is better, not in the home edge.
3. **What day 7 has to beat:** 0.641 log loss. Since a model trained on 2014-15 to 2019-20 learns the same old home edge, any gain will have to come from rest, form and ratings, not from fixing the intercept.